<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_2_Forest_Area_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ============================================================
# NOTEBOOK 2 — Forest Area Analysis & FAO Comparison
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [10]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q pandas matplotlib

#considering that output of notebook 1 are store
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# ── clone repo if not already cloned ─────────────────────
import os
if not os.path.exists('/content/Biochar_forest_estimation'):
    import getpass
    import subprocess
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git pull

/content/Biochar_forest_estimation
Updating 48d6314..96ad530
error: Your local changes to the following files would be overwritten by merge:
	__pycache__/data_config.cpython-312.pyc
Please commit your changes or stash them before you merge.
Aborting


In [12]:
## ── CELL 2: import Libraries and results & data─────────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from data_config import build_country_lookup, FAO_LSIB_REGION, country_thresholds, state_thresholds

print('✅ Libraries imported')
print('✅ Data config loaded')

✅ Libraries imported
✅ Data config loaded


In [13]:
## ── CELL 3  : merge results from notebook1 into main file ─────────────────────────────────────────────────
#define the folder in drive
GEE_folder = ('/content/drive/MyDrive/BiocharProject/GEE_exports/')
DATA_folder = '/content/Biochar_forest_estimation/data/'

#read the files created in notebook 1 for country
country_forest_area_10 = pd.read_csv(DATA_folder + 'forest_area_10.csv')
country_forest_area_20= pd.read_csv(DATA_folder + 'forest_area_20.csv')
country_forest_area_30 = pd.read_csv(DATA_folder + 'forest_area_30.csv')
country_forest_area_40 = pd.read_csv(DATA_folder + 'forest_area_40.csv')
country_forest_area_50  = pd.read_csv(DATA_folder + 'forest_area_50.csv')

#merge them and export the global file
country_total_forest_area =pd.concat([country_forest_area_10, country_forest_area_20, country_forest_area_30, country_forest_area_40, country_forest_area_50], ignore_index = True)
country_total_forest_area = country_total_forest_area.groupby(['country_na', 'threshold'])['sum'].sum().reset_index()
country_total_forest_area = country_total_forest_area.rename(columns={'country_na': 'country', 'sum': 'total_forest_area'})
country_total_forest_area.to_csv(DATA_folder + 'country_total_forest_area.csv', index=False)

#read the files created in notebook 1 for states USA
state_forest_area_10 = pd.read_csv(DATA_folder + 'states_forest_area_10.csv')
state_forest_area_20 = pd.read_csv(DATA_folder + 'states_forest_area_20.csv')
state_forest_area_30 = pd.read_csv(DATA_folder + 'states_forest_area_30.csv')
state_forest_area_40 = pd.read_csv(DATA_folder + 'states_forest_area_40.csv')
state_forest_area_50 = pd.read_csv(DATA_folder + 'states_forest_area_50.csv')

#merge them and export the global file
state_total_forest_area = pd.concat([state_forest_area_10, state_forest_area_20, state_forest_area_30, state_forest_area_40, state_forest_area_50], ignore_index = True)
state_total_forest_area = state_total_forest_area.groupby(['NAME', 'threshold'])['sum'].sum().reset_index()
state_total_forest_area = state_total_forest_area.rename(columns={'NAME': 'state','sum': 'total_forest_area'})
state_total_forest_area.to_csv(DATA_folder + 'state_total_forest_area.csv', index=False)

In [14]:
## ── CELL 4  : build region and subregion info for countries in 'country_total_forest_area' file by the intermidate of FAO_LSIB_REGION   ─────────────────────────────────────────────────
# 1. Generate the lookup dictionary
country_lookup = build_country_lookup(FAO_LSIB_REGION)

# 2. Create the new columns by looking up each country name
# we could actualy transform the country_lookup from a dictionary to a data frame then merge with df bsed onthe country name
country_total_forest_area  ['region'] = country_total_forest_area['country'].map(lambda l: country_lookup.get(l, {}).get('region', 'unknown'))
country_total_forest_area  ['subregion'] = country_total_forest_area['country'].map(lambda l: country_lookup.get(l, {}).get('subregion', 'unknown'))

print(country_total_forest_area.head(3))

       country  threshold  total_forest_area     region  subregion
0  Afghanistan         10           0.427305  Near East  West Asia
1  Afghanistan         20           0.287657  Near East  West Asia
2  Afghanistan         30           0.239637  Near East  West Asia


In [17]:
# ── CELL 6: Merge GEE results with FAO 2000 ──────────────
print(os.listdir('/content/Biochar_forest_estimation/data/'))
fao_fra_2025_data = pd.read_csv(DATA_folder + 'fao_fra_2025_data.csv')

FAO_forest_area_country_2000 = fao_fra_2025_data[['country', '2000']].rename(columns={'2000': 'FAO_2000_Mha'})

gee_fao_comparison = country_total_forest_area.merge(FAO_forest_area_country_2000, on='country', how='left')

print(gee_fao_comparison.head(40))


#counting th number of countries with data; since there is 5  different thresholds, nunique returns the number of unique values in the selected column:
print(f'Total countries: {gee_fao_comparison["country"].nunique()}')

# if mising, unique show the the values of unique values
missing = gee_fao_comparison[gee_fao_comparison['FAO_2000_Mha'].isna()]['country'].unique()
print(f"Total countries with no FAO match: {len(missing)}")
print(missing)

['gee_fao_comparison_region.csv', 'forest_area_40.csv', 'state_total_forest_area.csv', 'gee_fao_comparison.csv', 'states_forest_area_40.csv', 'states_forest_area_20.csv', 'states_forest_area_10.csv', '.gitkeep', 'forest_area_20.csv', 'states_forest_area_30.csv', 'gee_fao_comparison_world.csv', 'forest_area_30.csv', 'states_forest_area_50.csv', 'forest_area_10.csv', 'gee_fao_comparison_subregion.csv', 'forest_area_50.csv', 'country_total_forest_area.csv']


FileNotFoundError: [Errno 2] No such file or directory: '/content/Biochar_forest_estimation/data/fao_fra_2025_data.csv'

In [ ]:
# ── CELL 7: aggregate by subregion and region. ──────────────

gee_fao_comparison_subregion = gee_fao_comparison.groupby(['subregion', 'threshold']).agg(
    region=('region', 'first'), # Ensure 'region' is carried over
    total_forest_area=('total_forest_area', 'sum'),
    FAO_2000_Mha=('FAO_2000_Mha', 'sum')
).reset_index()

gee_fao_comparison_region = gee_fao_comparison_subregion.groupby(['region', 'threshold']).agg(
    total_forest_area=('total_forest_area', 'sum'),
    FAO_2000_Mha=('FAO_2000_Mha', 'sum')
).reset_index()

gee_fao_comparison_world = gee_fao_comparison_region.groupby(['threshold']).agg(
    total_forest_area=('total_forest_area', 'sum'),
    FAO_2000_Mha=('FAO_2000_Mha', 'sum')
).reset_index()

gee_fao_comparison_subregion.to_csv(DATA_folder + 'gee_fao_comparison_subregion.csv', index=False)
gee_fao_comparison_region.to_csv(DATA_folder + 'gee_fao_comparison_region.csv', index=False)
gee_fao_comparison_world.to_csv(DATA_folder + 'gee_fao_comparison_world.csv', index=False)

#print(gee_fao_comparison_subregion.head())
#print(gee_fao_comparison_region.head())
#print(gee_fao_comparison_world.head())

In [ ]:
# ── CELL 8: plot GEE vs FAO. ──────────────
#create the figure and axes

# add a devition column in gee_fao_comparison
gee_fao_comparison['deviation_Mha'] = gee_fao_comparison['total_forest_area'] - gee_fao_comparison['FAO_2000_Mha']
gee_fao_comparison['deviation_%'] = abs((gee_fao_comparison['deviation_Mha'] / gee_fao_comparison['FAO_2000_Mha'])/gee_fao_comparison['FAO_2000_Mha']).multiply(100)
clean_dev = gee_fao_comparison.replace([float('inf'), float('-inf')], float('nan'))
print(f"Mean deviation: {clean_dev['deviation_%'].median():.2f}%")

gee_fao_comparison.to_csv(DATA_folder + 'gee_fao_comparison.csv', index=False)

print(gee_fao_comparison.head())
fig, ax = plt.subplots(figsize=(10, 8))
plt.xlim(right = 1.05 *max(gee_fao_comparison['FAO_2000_Mha']))
plt.ylim(top = 1.05 *max(gee_fao_comparison['FAO_2000_Mha']))

#plot the dots
for threshold in country_thresholds:
  df =gee_fao_comparison[gee_fao_comparison['threshold'] == threshold]
  cleandev_df= clean_dev[clean_dev['threshold'] == threshold]
  x_values = df['FAO_2000_Mha'].dropna()   #check where are the nan in both column
  y_values = df['total_forest_area'].dropna()

  ax.scatter(x_values, y_values, label='GEE 10%')
  slope, intercept, r_value, p_value, std_err = stats.linregress(x_values, y_values)
  print(f'for threshold {threshold}%, R² = {r_value**2:.3f}, average deviation = {cleandev_df["deviation_%"].median():.2f}%')




#add labels, limit and title
ax.set_xlabel('FAO 2000 (Mha)')
ax.set_ylabel('GEE forest are (Mha)')
ax.set_title('GEE vs FAO forest area comparison')

match_max=max(gee_fao_comparison['FAO_2000_Mha'])

ax.plot([0, match_max], [0,match_max], linestyle = '--', color = 'black')

#show the plot
plt.show()

In [ ]:
print(os.listdir('/content/Biochar_forest_estimation/data/'))